# Model Training — Saturation Residual Head

**Purpose.** 585 train rows have `demand == 1.0`, concentrated in 23 Highway/4-5-lane geohashes ("saturating" cells). A regression model trained on the full distribution under-predicts saturation. A small binary classifier `P(demand >= 0.99 | features)` trained **only** on saturating geohashes can pull predictions toward 1.0 when it's confident.

**Inference rule.** For test rows whose geohash is in the saturating set AND `RoadType_modal == 'Highway'` AND `NumberofLanes_modal >= 2`:

  `pred_corrected = base_pred + P_sat * (1 - base_pred)`

(The other rows are left alone.) This is a *residual* correction — it only pulls predictions up toward 1.0; it never lowers them. It also degrades gracefully: low-confidence P_sat → tiny correction.

**Outputs**
- `oof/saturation_prob.parquet`         (train OOF P_sat, NaN outside saturating set)
- `predictions/saturation_prob_test.parquet`  (test P_sat)
- `models/saturation_fold{1..5}.txt`, `models/saturation_final.txt`


## 1. Setup + load

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import r2_score, roc_auc_score, log_loss

BASE = '.'
PROCESSED = os.path.join(BASE, 'processed')
OOF_DIR   = os.path.join(BASE, 'oof')
PRED_DIR  = os.path.join(BASE, 'predictions')
MODEL_DIR = os.path.join(BASE, 'models')
SEED = 42

train = pd.read_parquet(os.path.join(PROCESSED, 'train_features.parquet'))
test  = pd.read_parquet(os.path.join(PROCESSED, 'test_features.parquet'))
meta  = json.load(open(os.path.join(PROCESSED, 'feature_meta_v2.json')))
cv    = pd.read_parquet(os.path.join(PROCESSED, 'cv_folds.parquet'))
val_split = pd.read_parquet(os.path.join(PROCESSED, 'val_split_v2_indices.parquet'))
saturating_geohashes = set(json.load(open(os.path.join(PROCESSED, 'saturating_geohashes.json'))))

FEATS = meta['model_feature_cols']
CAT   = [c for c in meta['categorical_code_cols'] if c in FEATS]
SW    = meta['sample_weight_col']

train = train.merge(val_split.rename(columns={'row_id':'Index'}), on='Index', how='left')
train['is_val'] = (train['fold'] == 'val_fold').astype(np.int8)
train = train.merge(cv[['Index','fold_id']], on='Index', how='left').astype({'fold_id':'int8'})

print(f'Saturating geohashes: {len(saturating_geohashes)}')
sat_train_mask = train['geohash'].isin(saturating_geohashes)
sat_test_mask  = test ['geohash'].isin(saturating_geohashes)
print(f'Saturating train rows: {sat_train_mask.sum()}, test rows: {sat_test_mask.sum()}')

# Restrict to rows where the correction will fire: geohash ∈ sat AND Highway AND lanes≥2
fire_train = sat_train_mask & (train['RoadType_modal']=='Highway') & (train['NumberofLanes_modal']>=2)
fire_test  = sat_test_mask  & (test ['RoadType_modal']=='Highway') & (test ['NumberofLanes_modal']>=2)
print(f'Rows where correction fires: train {fire_train.sum()}, test {fire_test.sum()}')

Saturating geohashes: 23
Saturating train rows: 2414, test rows: 1081
Rows where correction fires: train 1890, test 846


## 2. Build binary target + train/val subsets

In [2]:
train['y_sat'] = (train['demand'] >= 0.99).astype(np.int8)
sub = train[sat_train_mask].copy()
print(f'Saturation classifier training pool: {len(sub)} rows')
print(f'Positive rate: {sub["y_sat"].mean():.3%}')
print(f'Positives per fold:'); print(sub.groupby('fold_id')['y_sat'].sum())

Saturation classifier training pool: 2414 rows
Positive rate: 24.689%
Positives per fold:
fold_id
1     68
2    186
3    244
4     43
5     55
Name: y_sat, dtype: int64


## 3. 5-fold CV training

In [3]:
base_params = dict(
    objective='binary', metric='auc',
    learning_rate=0.03, num_leaves=63, min_child_samples=10,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
    reg_alpha=0.1, reg_lambda=0.1,
    is_unbalance=True,
    verbosity=-1, random_state=SEED, n_jobs=-1,
)

oof_prob = np.full(len(train), np.nan)
test_acc = np.zeros(len(test))
fold_aucs = []
iters = []

for fid in [1,2,3,4,5]:
    in_fold = sub['fold_id'] == fid
    if in_fold.sum() == 0:
        print(f'Fold {fid}: empty, skip'); continue
    X_t = sub.loc[~in_fold, FEATS]; y_t = sub.loc[~in_fold, 'y_sat']
    X_v = sub.loc[ in_fold, FEATS]; y_v = sub.loc[ in_fold, 'y_sat']

    if y_t.nunique() < 2:
        print(f'Fold {fid}: train has single class, skip'); continue

    d_t = lgb.Dataset(X_t, label=y_t, categorical_feature=CAT, free_raw_data=False)
    d_v = lgb.Dataset(X_v, label=y_v, categorical_feature=CAT, reference=d_t, free_raw_data=False)
    m = lgb.train(base_params, d_t, num_boost_round=2000, valid_sets=[d_v],
                  callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)])
    p_v = m.predict(X_v, num_iteration=m.best_iteration)
    sub_idx = sub.index[in_fold]
    oof_prob[sub_idx] = p_v
    test_acc += m.predict(test[FEATS], num_iteration=m.best_iteration)
    m.save_model(os.path.join(MODEL_DIR, f'saturation_fold{fid}.txt'))
    try:
        auc = roc_auc_score(y_v, p_v) if y_v.nunique() == 2 else float('nan')
    except Exception:
        auc = float('nan')
    fold_aucs.append(auc)
    iters.append(m.best_iteration)
    print(f'Fold {fid}: AUC={auc:.4f}, best_iter={m.best_iteration}, fold_size={len(y_v)}')

test_cv_prob = test_acc / max(1, len(iters))

# Final refit on full saturating subset
d_full = lgb.Dataset(sub[FEATS], label=sub['y_sat'], categorical_feature=CAT, free_raw_data=False)
final = lgb.train(base_params, d_full, num_boost_round=int(np.mean(iters)) if iters else 800)
final.save_model(os.path.join(MODEL_DIR, 'saturation_final.txt'))
test_final_prob = final.predict(test[FEATS])
test_prob = 0.5*test_cv_prob + 0.5*test_final_prob

print(f'\nMean fold AUC: {np.nanmean(fold_aucs):.4f}')

Fold 1: AUC=0.9905, best_iter=472, fold_size=483
Fold 2: AUC=0.9843, best_iter=100, fold_size=483


Fold 3: AUC=0.9690, best_iter=15, fold_size=483


Fold 4: AUC=0.9980, best_iter=115, fold_size=483
Fold 5: AUC=0.9940, best_iter=5, fold_size=482



Mean fold AUC: 0.9872


## 4. Sanity check on val_fold subset

In [4]:
val_mask = (train['is_val']==1) & sat_train_mask
if val_mask.sum() > 0 and train.loc[val_mask, 'y_sat'].nunique() == 2:
    auc_val = roc_auc_score(train.loc[val_mask, 'y_sat'].values, oof_prob[val_mask.values])
    print(f'val_fold subset AUC (saturating geohashes only): {auc_val:.4f}')
    print(f'val_fold positives: {train.loc[val_mask, "y_sat"].sum()} / {val_mask.sum()}')
else:
    print('val_fold subset too small or single-class for AUC')

val_fold subset AUC (saturating geohashes only): 0.9627
val_fold positives: 426 / 1081


## 5. Save outputs

In [5]:
# Build a row-aligned OOF series: NaN outside saturating set, P_sat inside
# Important: we DON'T fire on rows that aren't Highway+lanes>=2 — but P_sat is still computed
oof_out = pd.DataFrame({'Index': train['Index'].values, 'sat_prob_oof': oof_prob})
test_out = pd.DataFrame({'Index': test['Index'].values, 'sat_prob_test': test_prob})

# For test rows NOT in saturating set, set sat_prob_test to NaN (so blend code can detect)
test_out.loc[~sat_test_mask.values, 'sat_prob_test'] = np.nan

oof_out.to_parquet(os.path.join(OOF_DIR, 'saturation_prob.parquet'), index=False)
test_out.to_parquet(os.path.join(PRED_DIR, 'saturation_prob_test.parquet'), index=False)

# Also persist the "fire" masks for use in blending
np.save(os.path.join(OOF_DIR, 'sat_fire_mask_train.npy'), fire_train.values)
np.save(os.path.join(PRED_DIR, 'sat_fire_mask_test.npy'), fire_test.values)

print('Saved:')
print(f'  oof/saturation_prob.parquet                (oof N={(~oof_out["sat_prob_oof"].isna()).sum()})')
print(f'  predictions/saturation_prob_test.parquet   (test N={(~test_out["sat_prob_test"].isna()).sum()})')
print(f'  oof/sat_fire_mask_train.npy                (sum={fire_train.sum()})')
print(f'  predictions/sat_fire_mask_test.npy         (sum={fire_test.sum()})')

Saved:
  oof/saturation_prob.parquet                (oof N=2414)
  predictions/saturation_prob_test.parquet   (test N=1081)
  oof/sat_fire_mask_train.npy                (sum=1890)
  predictions/sat_fire_mask_test.npy         (sum=846)
